In [1]:
import pyiceberg

In [ ]:
NESSIE_URI = "http://localhost:19120/iceberg"
MINIO_ENDPOINT = "http://localhost:9000"

In [2]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "nessie",
    **{
        "type": "rest",
        "uri": "http://localhost:19120/iceberg",
        "warehouse": "s3://iceberg-warehouse",
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "lakeadmin",
        "s3.secret-access-key": "Pass@12345",
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print(catalog.list_namespaces())

[('demo',)]


In [5]:
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, IntegerType, StringType

schema = Schema(
    NestedField(1, "id", IntegerType(), required=True),
    NestedField(2, "message", StringType(), required=True),
)

table = catalog.create_table_if_not_exists(
    "demo.events",
    schema=schema,
)

print("Location:", table.location())
print("Schema:", table.schema())
print("Properties:", table.properties)

Location: s3://iceberg-warehouse/demo/events_b671920b-8d39-4cd8-ad1d-e1c087f39ade
Schema: table {
  1: id: required int
  2: message: required string
}
Properties: {'nessie.catalog.content-id': 'e78e2f93-4201-40db-921a-3629b4627e65', 'created-at': '2026-09-13T00:12:09.442356703Z', 'nessie.commit.id': '30446cb6723e965a90c0b36487f8dedcb341eb2e0d1ca1ed209594fffa0e9ac0', 'gc.enabled': 'false', 'nessie.commit.ref': 'main'}


In [6]:
catalog.list_tables("demo")

[('demo', 'events')]

In [7]:
table = catalog.load_table("demo.events")

print(table.location())
print(table.schema())

s3://iceberg-warehouse/demo/events_b671920b-8d39-4cd8-ad1d-e1c087f39ade
table {
  1: id: required int
  2: message: required string
}


In [ ]:
import pyarrow as pa

arrow_schema = pa.schema([
    pa.field("id", pa.int32(), nullable=False),
    pa.field("message", pa.string(), nullable=False),
])

data = pa.Table.from_pydict(
    {
        "id": [1, 2, 3],
        "message": ["hello", "iceberg", "nessie"],
    },
    schema=arrow_schema,
)

table.append(data)

print("Rows appended successfully")

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                 ┃ Dataframe field             ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: id: required int         │ 1: id: optional long        │
│ ❌ │ 2: message: required string │ 2: message: optional string │
└────┴─────────────────────────────┴─────────────────────────────┘

ValueError: Mismatch in fields:
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃    ┃ Table field                 ┃ Dataframe field             ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ❌ │ 1: id: required int         │ 1: id: optional long        │
│ ❌ │ 2: message: required string │ 2: message: optional string │
└────┴─────────────────────────────┴─────────────────────────────┘
